In [1]:
from copy import copy
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from scipy.optimize import bisect, brentq
from scipy.stats import norm

In [2]:
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

In [3]:
class GBMdynamics:

    def __init__(self, X, r, rGrow, sigma=None):
        self.X = X
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [4]:
class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

    def BSvega(self, dynamics, contract):
        # Returns time-$0$ vega

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        return np.exp(-dynamics.r*contract.T)*F*norm.pdf(d1)*np.sqrt(contract.T)

    def IV(self, dynamics, contract):
        # ignores dynamics.sigma, because this function solves for sigma.
        # Returns time-$0$ implied volatility

        if contract.price is None:
            raise ValueError('Contract price must be given')

        df = np.exp(-dynamics.r*contract.T)
        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        lowerbound = np.max([0, df*(F - contract.K)])
        C = contract.price
        if C < lowerbound:
            return np.nan
        if C == lowerbound:
            return 0
        if C >= F*df:
            return np.nan

        dynamics_try = copy(dynamics)

        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2
        hi = sigma_try
        lo = hi/2

        impliedVolatility = brentq(
            lambda sigma: self.BSpriceCall(dynamics_try.update_sigma(sigma), contract) - C,
            lo,
            hi
        )

        return impliedVolatility

## Problem 1

In [5]:
options = pd.read_csv('hw1-rut.csv')
options.set_index('Strike', drop=False, inplace=True)

T = 25/365
r = 0.0432

options['CallMid'] = (options['CallBid'] + options['CallAsk']) / 2
options['PutMid'] = (options['PutBid'] + options['PutAsk']) / 2

options[['Strike', 'CallMid', 'PutMid']].head()

,Strike,CallMid,PutMid
Strike,,,
1750,1750,515.35,1.200
1760,1760,505.45,1.250
1770,1770,495.50,1.300
1780,1780,485.60,1.375
1790,1790,475.70,1.450


In [6]:
# Calculate the forward price.
options['MidDiff'] = options['CallMid'] - options['PutMid']
K_star = options.loc[options['MidDiff'].abs().idxmin(), 'Strike']
mid_diff_star = options.loc[options['MidDiff'].abs().idxmin(), 'MidDiff']
F = K_star + np.exp(r*T) * mid_diff_star

pd.Series({'K_star': K_star, 'F0': F})

K_star    2265.000000
F0        2265.401185
dtype: float64

In [7]:
options['ImpliedCallMid'] = options['PutMid'] + np.exp(-r*T) * (F - options['Strike'])

options[['Strike', 'PutMid', 'ImpliedCallMid']].head()

,Strike,PutMid,ImpliedCallMid
Strike,,,
1750,1750,1.200,515.078417
1760,1760,1.250,505.157962
1770,1770,1.300,495.237507
1780,1780,1.375,485.342053
1790,1790,1.450,475.446598


In [8]:
options['CallOrImpliedCall'] = np.where(options['Strike'] > F, options['CallMid'], options['ImpliedCallMid'])

options[['Strike', 'CallMid', 'ImpliedCallMid', 'CallOrImpliedCall']].head()

,Strike,CallMid,ImpliedCallMid,CallOrImpliedCall
Strike,,,,
1750,1750,515.35,515.078417,515.078417
1760,1760,505.45,505.157962,505.157962
1770,1770,495.50,495.237507,495.237507
1780,1780,485.60,485.342053,485.342053
1790,1790,475.70,475.446598,475.446598


In [9]:
hw1analytic = AnalyticEngine()
dynamics = GBMdynamics(X=F, r=r, rGrow=0, sigma=None)

In [10]:
options['IV'] = options.apply(lambda row: hw1analytic.IV(dynamics,  CallOption(K=row.Strike, T=T, price=row.CallOrImpliedCall)), axis=1)

In [11]:

fig = go.Figure()

fig.add_trace(go.Scatter(x=options.Strike, y=options.IV, mode='markers', name='Implied Volatility'))

fig.update_layout(title='Implied Volatility vs. Strike',
                  xaxis_title='Strike Price',
                  yaxis_title='Implied Volatility')
fig.show()


## Problem 2

In [12]:
k = np.log(options.Strike / F)

coefficients = np.polyfit(k, options.IV, 4)
polynomial = np.poly1d(coefficients)

alpha = pd.Series(
    {
        'alpha0': coefficients[4],
        'alpha1': coefficients[3],
        'alpha2': coefficients[2],
        'alpha3': coefficients[1],
        'alpha4': coefficients[0],
    }
)

alpha

alpha0     0.218082
alpha1    -0.460467
alpha2     2.361710
alpha3     7.057827
alpha4    20.110443
dtype: float64

In [13]:
k_grid = np.linspace(k.min(), k.max(), 100)
y_fit = polynomial(k_grid)

fig = go.Figure()

fig.add_trace(go.Scatter(x=k, y=options.IV, mode='markers', name='Observed midpoint implied volatility'))
fig.add_trace(go.Scatter(x=k_grid, y=y_fit, mode='lines', name='Fitted polynomial parameterization'))

fig.update_layout(title='Implied Volatility vs log-moneyness',
                  xaxis_title='log-moneyness',
                  yaxis_title='Implied Volatility')

fig.show()

In [14]:
options['IV_fit'] = polynomial(k)
options['IV_residual'] = options['IV'] - options['IV_fit']

overpricedStrike = options.loc[options['IV_residual'].idxmax(), 'Strike']
overpricedStrike

np.int64(2570)

In [ ]:
# The vega calculation for each strike uses the fitted polynomial implied vol for that strike
options['vega'] = options.apply(
    lambda row: hw1analytic.BSvega(
        GBMdynamics(
            X=F,
            r=r,
            rGrow=0,
            sigma=polynomial(np.log(row.Strike / F))
        ),
        CallOption(K=row.Strike, T=T)
    ),
    axis=1
)

options[['Strike', 'vega']].head()

,Strike,vega
Strike,,
1750,1750,21.227648
1760,1760,21.492620
1770,1770,21.834532
1780,1780,22.255869
1790,1790,22.759652


In [16]:
options['skewsensitivity'] = options['vega'] * np.log(options['Strike'] / F)

options[['Strike', 'skewsensitivity']].head()

,Strike,skewsensitivity
Strike,,
1750,1750,-5.479622
1760,1760,-5.425555
1770,1770,-5.388158
1780,1780,-5.366747
1790,1790,-5.360723


### Problem 2(c)

Holding $k$ fixed, the fitted skew is

$$
\sigma_p(k)=\alpha_0+\alpha_1 k+\alpha_2 k^2+\alpha_3 k^3+\alpha_4 k^4.
$$

So

$$
\frac{\partial \sigma_p(k)}{\partial \alpha_1}=k.
$$

Since the price sensitivity to volatility is vega, the sensitivity of the time-0 call or put price to $\alpha_1$ is

$$
\frac{\partial \text{Price}_0}{\partial \alpha_1}
=
\text{Vega}_0 \cdot \frac{\partial \sigma_p(k)}{\partial \alpha_1}
=
e^{-rT}F_0\sqrt{T}\,N'(d_1)\,k,
$$

where

$$
d_1=\frac{-k}{\sigma_p(k)\sqrt{T}}+\frac{\sigma_p(k)\sqrt{T}}{2}.
$$

Thus, for fixed $k$, call and put sensitivities are the same:
$$
\frac{\partial \text{Price}_0}{\partial \alpha_1}=e^{-rT}F_0\sqrt{T}\,N'(d_1)\,k.
$$


In [17]:
import plotly.express as px

fig = px.line(options, x="Strike", y="skewsensitivity", title='Sensitivity to Skew Slope, vs. Strike')
fig.show()

In [18]:
whichContractGainsMostFromSlopeIncrease = options.loc[options['skewsensitivity'].idxmax(), 'Strike']
whichContractGainsMostFromSlopeIncrease

np.int64(2385)

In [19]:
whichContractLosesMostFromSlopeIncrease = options.loc[options['skewsensitivity'].idxmin(), 'Strike']
whichContractLosesMostFromSlopeIncrease

np.int64(2100)

In [20]:
summary = pd.Series(
    {
        'F0': F,
        'K_star': K_star,
        'overpricedStrike': overpricedStrike,
        'gainsMostWhenAlpha1Increases': whichContractGainsMostFromSlopeIncrease,
        'losesMostWhenAlpha1Increases': whichContractLosesMostFromSlopeIncrease,
    }
)
summary

F0                              2265.401185
K_star                          2265.000000
overpricedStrike                2570.000000
gainsMostWhenAlpha1Increases    2385.000000
losesMostWhenAlpha1Increases    2100.000000
dtype: float64